# MCP(Model Context Protocol)

### MCP(Model Context Protocol) 정의

**MCP**는 **AI 애플리케이션과 외부 도구 간의 상호작용을 표준화**한 오픈 프로토콜입니다.  
즉, LLM이 다양한 외부 시스템(API, 데이터베이스, 도구 등)과 일관된 방식으로 통신할 수 있도록 하는 **표준 인터페이스 계층**입니다.

---

### MCP의 4가지 핵심 특징

| 특징 | 설명 | 장점 | 예시 |
|------|------|------|------|
| **표준화된 도구 인터페이스** | 모든 외부 도구가 동일한 프로토콜로 연결 | 새로운 도구 추가 시 별도 학습 불필요 | 날씨 API, DB, 파일 시스템 등 동일 방식 호출 |
| **다양한 전송 메커니즘** | **공식: `stdio`, `Streamable HTTP` (구 `HTTP+SSE`는 deprecated)** | 환경(로컬/원격)에 맞는 최적 방식 선택 | 로컬은 `stdio`, 원격은 `Streamable HTTP` |
| **동적 도구 검색** | 런타임 시 사용 가능한 도구 자동 탐색 | 코드 수정 없이 확장 가능 | 새로운 플러그인 자동 인식 |
| **확장 가능한 아키텍처** | 모듈형 구조 및 멀티 서버 지원 | 시스템 규모 확장에 유연 | 필요 기능만 선택적 추가 |

---

### MCP 전송 메커니즘 비교

| 구분 | **stdio** | **HTTP+SSE (레거시)** | **Streamable HTTP (현행)** |
|------|------------|------------------------|-----------------------------|
| **전송 계층** | 표준입출력 파이프(로컬 IPC) | GET(SSE 수신) + POST(송신) 2 엔드포인트 | 단일 HTTP 엔드포인트(POST/GET), **SSE로 응답 스트리밍** |
| **통신 구조** | 한 프로세스 내 파이프 기반 | S→C는 SSE, C→S는 POST로 분리 | C→S: POST / S→C: **SSE 스트림** → 실질적 양방향 통신 |
| **실시간성 / 재개 지원** | 빠름(로컬 IPC 수준), 재개 불가 | 단방향 스트림만 가능, 재개 미지원 | **Event ID / `Last-Event-ID` 기반 스트림 재개 가능** |
| **적합 환경** | 로컬 도구·IDE 플러그인 | 과거 원격 MCP 서버 호환 | **최신 원격 MCP 서버 / 실시간 스트리밍 서비스** |
| **표준 지위** | 지속 지원 | **Deprecated (2025-03-26)** | **Current Standard (Streamable HTTP)** |

> ⚙️ **보안 주의사항:**  
> - Streamable HTTP 사용 시 **Origin 검증 및 로컬(127.0.0.1) 바인딩** 권장  
> - 외부 네트워크 통신 시 **토큰 기반 인증(OAuth 등)** 필수  
> - HTTP+SSE는 호환성 목적 외에는 더 이상 권장되지 않음

---

### 요약

- **stdio** → 로컬 환경용, 빠르고 안정적  
- **Streamable HTTP** → 현대적 원격 통신 표준, SSE 기반 실시간 양방향 스트리밍 지원  
- **HTTP+SSE** → 과거 임시 방식으로 사용되었으나 **2025년 3월 이후 폐기됨**

---

### 핵심 포인트

> 🔎 **MCP는 "AI와 도구를 연결하는 표준 인터페이스"**  
> - 도구 검색, 요청·응답 포맷, 스트리밍, 인증·보안 정책을 통합적으로 관리  
> - 로컬(예: IDE 확장)에는 `stdio`,  
>   원격(예: 클라우드 도구 서버)에는 `Streamable HTTP`를 사용하는 것이 권장됨.


In [1]:
# 비동기 처리를 위한 nest_asyncio (Jupyter 환경에서 필수)
import nest_asyncio
from typing import List, Dict, Any

# OpenAI LLM 모델 사용을 위한 import
from langchain_openai import ChatOpenAI

# LangGraph의 사전 구축된 React 에이전트 import
from langgraph.prebuilt import create_react_agent

# 메모리 기반 체크포인터 (대화 상태 저장)
from langgraph.checkpoint.memory import InMemorySaver

# MCP 클라이언트 - 여러 MCP 서버를 동시에 관리하는 핵심 컴포넌트
from langchain_mcp_adapters.client import MultiServerMCPClient

In [2]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv
import os

# API KEY 정보로드
load_dotenv(override=True)

# 비동기 호출 활성화
nest_asyncio.apply()

In [3]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("LangChain-Tutorial")

LangSmith 추적을 시작합니다.
[프로젝트명]
LangChain-Tutorial


## MCP 다양한 서버 제작 실습

### 서버 파일 구성 및 특징

이 튜토리얼에서는 세 가지 타입의 **MCP 서버**를 사용합니다.  
각 서버는 전송 방식과 역할이 다르며, `MultiServerMCPClient`로 통합 관리할 수 있습니다.

| 서버 유형 | 파일명 | 전송 방식 | 주요 기능 | 사용 시나리오 |
|-----------|---------|------------|------------|----------------|
| **로컬 서버** | `mcp_server_local.py` | stdio | 파일 관리, 로컬 데이터 조회 | 빠른 응답이 필요한 기본 기능 |
| **원격 서버** | `mcp_server_remote.py` | Streamable HTTP | 시간 조회, 외부 API 연동 | 네트워크를 통한 원격 서비스 |
| **검색 서버** | `mcp_server_rag.py` | stdio | RAG 문서 검색, 벡터 유사도 검색 | 대용량 문서에서 정확한 정보 검색 |

> 💡 **참고:** MCP 공식 스펙 기준으로 `WebSocket`은 지원 예시로 언급될 수 있지만,  
> 실제 표준 전송 방식은 `stdio`와 `Streamable HTTP` 두 가지입니다.  
> (`HTTP+SSE`는 2025년 3월부로 폐기됨)

---

### 서버별 상세 특징

#### 🖥️ 로컬 서버 (`stdio`)
- **장점**  
  - 빠른 응답속도  
  - 안정적 연결 (네트워크 불필요)  
  - OS 파일 접근이 용이  
- **단점**  
  - 동일 머신 내에서만 실행 가능  
- **적용 분야**  
  - 로컬 파일 시스템 제어, 오프라인 데이터 처리, IDE 통합 도구

---

#### ☁️ 원격 서버 (`Streamable HTTP`)
- **장점**  
  - 네트워크를 통한 원격 호출 가능  
  - 확장성과 보안 설정 용이 (토큰 인증, Origin 검증 등)  
  - SSE 기반 **스트리밍 응답 및 재개 지원**  
- **단점**  
  - 네트워크 지연 및 연결 관리 필요  
- **적용 분야**  
  - 외부 API 연동, 클라우드 서비스 연결, 실시간 스트리밍 응답 제공

---

#### 🔍 검색 서버 (`stdio + RAG`)
- **장점**  
  - 대용량 문서 처리 및 의미 기반 검색 지원  
  - 벡터 유사도 검색으로 정교한 컨텍스트 제공  
- **단점**  
  - 초기 임베딩 구축 및 리소스 사용량 높음  
- **적용 분야**  
  - 사내 지식베이스 검색, 문서 QA 시스템, LLM 보조 검색 엔진

---

### 서버 분리의 장점

| 장점 | 설명 | 실제 적용 예시 |
|------|------|----------------|
| **모듈성** | 각 기능을 독립적으로 개발·배포 가능 | 날씨 서버 오류 시에도 시간 서버는 정상 작동 |
| **확장성** | 필요한 기능만 선택적으로 추가 가능 | 번역 서버 추가 시 기존 서버 영향 없음 |
| **안정성** | 장애가 다른 서버로 전이되지 않음 | 검색 서버 장애 시에도 나머지 서비스 유지 |
| **성능 최적화** | 서버별 리소스 설정 조정 가능 | 검색 서버는 GPU, 시간 서버는 CPU 경량 운영 |

## MultiServerMCPClient

**`MultiServerMCPClient`** 는 **여러 MCP 서버를 하나의 통합 인터페이스로 관리**하는 핵심 컴포넌트입니다.  
각 서버(`stdio`, `Streamable HTTP`)에서 제공하는 도구들을 자동으로 수집하고,  
단일 클라이언트에서 일관된 방식으로 호출할 수 있도록 설계되어 있습니다.

---

### MultiServerMCPClient의 핵심 기능

| 기능 | 설명 | 코드 예시 | 장점 |
|------|------|-----------|------|
| **서버 구성 관리** | 각 MCP 서버 설정을 딕셔너리로 정의·관리 | `server_configs = {"weather": {...}}` | 설정 변경 및 버전 관리 용이 |
| **동적 도구 로딩** | 각 서버가 제공하는 도구를 자동 탐색·통합 | `tools = await client.get_tools()` | 코드 수정 없이 실시간 도구 업데이트 |
| **통합 인터페이스** | 모든 서버의 도구를 하나의 통합 리스트로 제공 | `for tool in tools: result = await tool.invoke()` | 서버별 호출 방식 차이 제거 |
| **에러 처리 및 복구** | 특정 서버 장애 시 다른 서버는 정상 작동 | 내부 재연결 및 로깅 처리 | 시스템 안정성 향상 |

---

### 서버 구성 설정 상세

#### 기본 구성 형태
```python
server_configs = {
    "서버명": {
        "command": "실행명령어",      # uv, python, node 등
        "args": ["인자1", "인자2"],   # 실행 시 전달할 인자
        "transport": "전송방식",      # stdio 또는 streamable_http
    }
}
```

📘 stdio 전송 방식 예시
```python
"local_server": {
    "command": "uv",
    "args": ["run", "python", "server/mcp_server_local.py"],
    "transport": "stdio",
}
```

🌐 Streamable HTTP 전송 방식 예시
```python
"remote_server": {
    "url": "http://127.0.0.1:8002/mcp",
    "transport": "streamable_http",
}
```

In [4]:
# MultiServerMCPClient 설정 예제
async def setup_mcp_client(server_configs: List[Dict[str, Any]]):
    """
    MCP 클라이언트를 설정하고 도구를 가져옵니다.

    Args:
        server_configs: 서버 설정 정보 리스트

    Returns:
        tuple: (MCP 클라이언트, 도구 리스트)
    """

    # MultiServerMCPClient 인스턴스 생성 - 여러 MCP 서버를 통합 관리
    client = MultiServerMCPClient(server_configs)

    # 모든 연결된 서버로부터 사용 가능한 도구들을 수집
    tools = await client.get_tools()

    # 로드된 도구 정보를 콘솔에 출력 (디버깅 및 확인용)
    print(f"✅ {len(tools)} 개의 MCP 도구가 로드되었습니다:")
    for tool in tools:
        print(f"  - {tool.name}")  # 각 도구의 이름 출력

    return client, tools

### 로컬 MCP 서버 방식 사용

In [5]:
import sys
from pathlib import Path

# [1] 서버 .py 파일이 어디 있는지 찾기
#     폴더를 어떻게 열었는지에 따라 경로가 달라서, 자주 쓰는 두 가지만 순서대로 확인합니다.
server_script = Path("02-MCP/server/mcp_server_local.py").resolve()  # 프로젝트 루트를 연 경우
if not server_script.is_file():
    server_script = Path("server/mcp_server_local.py").resolve()  # 02-MCP 폴더만 연 경우
if not server_script.is_file():
    raise FileNotFoundError("mcp_server_local.py를 찾을 수 없습니다. mysuni_aiagent 또는 02-MCP 폴더를 연 뒤 다시 실행하세요.")

# [2] 어떤 Python으로 서버를 실행할지
#     "python"이라고만 쓰면 노트북과 다른 Python이 잡혀서 서버가 바로 꺼질 수 있습니다.
#     sys.executable = 지금 이 노트북이 돌아가는 Python (fastmcp가 설치된 그 환경)
server_configs = {
    "weather": {
        "command": sys.executable,
        "args": [str(server_script)],
        "transport": "stdio",
    },
}

client, tools = await setup_mcp_client(server_configs=server_configs)

✅ 1 개의 MCP 도구가 로드되었습니다:
  - get_weather


In [6]:
# OpenRouter를 통한 GPT 모델 사용 설정
llm = ChatOpenAI(
    model="gpt-4.1",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
)  # 일관된 결과를 위해 temperature=0

# LangGraph의 사전 구축된 React Agent 생성
# - llm: 사용할 언어 모델
# - tools: MCP에서 로드한 도구들
# - checkpointer: 대화 상태를 메모리에 저장하여 연속 대화 지원
agent = create_react_agent(
    llm, tools, checkpointer=InMemorySaver()  # 상태 저장을 위한 체크포인터
)

/var/folders/fn/cyp1s2wj7c39s35jrjsksnkw0000gn/T/ipykernel_10896/49780026.py:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [7]:
# 스트리밍 출력과 랜덤 UUID 생성을 위한 유틸리티 함수 import
from langchain_teddynote.messages import astream_graph, random_uuid
from langchain_core.runnables import RunnableConfig

# 대화 스레드를 구분하기 위한 고유 ID 생성
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 에이전트 실행 - 서울 날씨 조회 요청
# astream_graph: 그래프 실행 결과를 스트리밍으로 출력 (실시간으로 처리 과정 확인 가능)
response = await astream_graph(
    agent,  # 실행할 에이전트
    inputs={
        "messages": [("human", "안녕하세요. 서울의 날씨를 알려주세요.")]
    },  # 사용자 입력 메시지
    config=config,  # 대화 설정 (스레드 ID 포함)
)


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
It's always Sunny in 서울
🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
서울의 날씨는 항상 맑음(Sunny)입니다! 오늘도 좋은 하루 보내세요.

### Streamable HTTP 전송 방식 사용

`mcp_server_remote.py`는 **127.0.0.1:8002** 에서 **Streamable HTTP** MCP(`/mcp`)를 띄웁니다. 아래 코드 셀에서 서버를 백그라운드로 실행한 뒤, 그 다음 셀에서 클라이언트를 연결하세요.

터미널에서 직접 띄울 때는 **노트북과 같은 Python**을 쓰는 것이 좋습니다.

프로젝트 루트(`mysuni_aiagent`)를 연 경우:

```bash
python 02-MCP/server/mcp_server_remote.py
```

`02-MCP` 폴더만 연 경우:

```bash
python server/mcp_server_remote.py
```

In [8]:
import subprocess
import sys
import time
from pathlib import Path

# 로컬 날씨 서버 셀과 같이, 폴더를 어떻게 열었는지에 따라 경로만 바꿉니다.
remote_script = Path("02-MCP/server/mcp_server_remote.py").resolve()
if not remote_script.is_file():
    remote_script = Path("server/mcp_server_remote.py").resolve()
if not remote_script.is_file():
    raise FileNotFoundError(
        "mcp_server_remote.py를 찾을 수 없습니다. mysuni_aiagent 또는 02-MCP 폴더를 연 뒤 다시 실행하세요."
    )

# MCP 서버를 백그라운드에서 실행합니다. (uv 대신 노트북과 동일한 Python)
# stdout/stderr를 PIPE로 두면 로그가 쌓여 서버가 멈출 수 있어 DEVNULL로 버립니다.
mcp_server_process = subprocess.Popen(
    [sys.executable, str(remote_script)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# uvicorn 기동까지 잠시 대기 (이미 8002를 쓰는 프로세스가 있으면 실패할 수 있음)
time.sleep(3)

# 종료시
# mcp_server_process.terminate()

In [9]:
# HTTP 기반 MCP 서버 설정 예제 (바로 위 셀에서 서버가 떠 있어야 연결됩니다)
http_server_config = {
    "current_time": {
        "url": "http://127.0.0.1:8002/mcp",  # 원격 MCP 서버의 HTTP 엔드포인트
        "transport": "streamable_http",  # HTTP 스트리밍 방식으로 통신
    },
}

# HTTP MCP 서버에 연결하여 클라이언트 생성
client, http_tools = await setup_mcp_client(server_configs=http_server_config)

✅ 2 개의 MCP 도구가 로드되었습니다:
  - get_current_time
  - retrieve


In [10]:
# HTTP 서버용 LLM 설정
llm = ChatOpenAI(
    model="gpt-4.1",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
)  # 일관성을 위해 동일한 모델 사용

# HTTP 도구를 사용하는 React Agent 생성
agent = create_react_agent(
    llm,
    http_tools,
    checkpointer=InMemorySaver(),  # HTTP로 로드된 도구들과 메모리 저장소 사용
)

/var/folders/fn/cyp1s2wj7c39s35jrjsksnkw0000gn/T/ipykernel_10896/729522017.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [11]:
# 새로운 스레드로 HTTP 서버와 상호작용
from langchain_teddynote.messages import astream_graph, random_uuid
from langchain_core.runnables import RunnableConfig

# 새로운 대화 세션을 위한 고유 ID 생성
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# HTTP MCP 서버를 통해 현재 시간 조회 요청
response = await astream_graph(
    agent,  # HTTP 도구가 연결된 에이전트
    inputs={
        "messages": [("human", "안녕하세요. 현재 시간을 알려주세요.")]
    },  # 시간 조회 요청
    config=config,  # 대화 세션 설정
)


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
2026-04-22 12:05:02 KST (UTC+0900)
🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
안녕하세요! 현재 시간은 2026년 4월 22일 12시 5분입니다.

### RAG MCP 활용 예제

In [12]:
import sys
from pathlib import Path

# RAG 전용 MCP(stdio). 위 날씨/원격 셀과 같이 경로·Python을 맞춥니다.
rag_script = Path("02-MCP/server/mcp_server_rag.py").resolve()
if not rag_script.is_file():
    rag_script = Path("server/mcp_server_rag.py").resolve()
if not rag_script.is_file():
    raise FileNotFoundError(
        "mcp_server_rag.py를 찾을 수 없습니다. mysuni_aiagent 또는 02-MCP 폴더를 연 뒤 다시 실행하세요."
    )

rag_server_config = {
    "rag": {
        "command": sys.executable,
        "args": [str(rag_script)],
        "transport": "stdio",
    },
}

client, rag_tools = await setup_mcp_client(server_configs=rag_server_config)

✅ 1 개의 MCP 도구가 로드되었습니다:
  - retrieve


In [13]:
# RAG 서버용 LLM 설정 (OpenRouter 사용)
llm = ChatOpenAI(
    model="gpt-4.1",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
)  # 정확한 검색 결과를 위해 temperature=0

# RAG 도구를 활용하는 React Agent 생성
rag_agent = create_react_agent(
    llm, rag_tools, checkpointer=InMemorySaver()  # RAG 검색 도구와 메모리 저장소 연결
)

/var/folders/fn/cyp1s2wj7c39s35jrjsksnkw0000gn/T/ipykernel_10896/3006759139.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  rag_agent = create_react_agent(


In [14]:
# RAG 에이전트를 사용한 문서 검색 예제
from langchain_teddynote.messages import astream_graph, random_uuid
from langchain_core.runnables import RunnableConfig

# RAG 전용 대화 세션 생성
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 삼성 가우스 AI에 대한 정보를 MCP RAG 서버에서 검색
# MCP 서버를 명시적으로 사용하도록 요청하여 정확한 정보 검색 유도
_ = await astream_graph(
    rag_agent,  # RAG 검색 도구가 연결된 에이전트
    inputs={
        "messages": [
            (
                "human",
                "미드저니 V1에 대한 내용을 검색해 주세요.",
            )
        ]
    },
    config=config,  # RAG 세션 설정
)


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
SPRi AI Brief 2025년 8월호
미드저니, 첫 번째 비디오 생성 AI 모델 ‘V1’ 출시
KEY Contents
n 미드저니가 1회 작업으로 5초 길이의 동영상 4개를 제작할 수 있는 비디오 생성 모델 ‘V1’을
출시하고 여타 비디오 생성 모델보다 25배 이상 저렴한 가격을 책정했다고 강조
n 미드저니는 실시간 오픈월드 시뮬레이션이 가능한 AI 모델 개발을 궁극적 목표로 제시하고,
2026년에 3D 모델과 실시간 처리 모델을 출시한 뒤 각 모델을 하나로 통합할 계획이라고 설명
£ ‘V1’, 미드저니에서 생성한 이미지나 외부 이미지를 동영상으로 변환
n AI 이미지 생성 플랫폼 미드저니(Midjourney)가 2025년 6월 19일 비디오 생성 모델 ‘V1’을 출시
∙ V1은 이미지를 동영상으로 변환하는 모델로, 미드저니 플랫폼에서 제작된 이미지나 외부 이미지를
바탕으로 동영상을 생성하며, ‘자동’ 설정 시에는 모션 프롬프트가 자동으로 생성되고 ‘수동’ 설정을
선택하면 사용자 지시에 따라 사물이나 장면의 움직임을 생성
∙ 사용자는 움직임 강도를 피사체와 카메라가 모두 움직이는 ‘하이 모션(High Motion)’과 카메라는 거의
고정되어 있고 피사체가 천천히 움직이도록 연출 ‘로우 모션(Low Motion)’ 중에서 선택 가능
∙ 웹 전용 모델인 V1은 1회 동영상 생성 작업으로 5초 길이의 동영상 4개를 제작하며, 생성된 동영상은
한 번에 4초씩 최대 4회까지 영상 길이를 확장할 수 있도록 허용
∙ 미드저니는 동영상 생성 작업에 이미지 작업보다 약 8배의 비용을 책정했으며, 이는 동영상 1초당 이미지
1장의 비용에 해당한다며 시장에 출시된 기존 제품 대비 25배 이상 저렴하다고 주장*
* GPU 제공 시간에 따

## React Agent와 MCP 통합

React Agent는 추론(Reason)과 행동(Act)을 반복하는 패턴을 구현합니다. MCP 도구와 함께 사용하면 강력한 에이전트를 만들 수 있습니다.

In [15]:
# React Agent와 MCP를 통합하는 함수 정의
async def create_mcp_react_agent(server_configs: dict):
    """
    MCP 도구를 사용하는 React Agent를 생성합니다.

    Args:
        server_configs: MCP 서버 설정 딕셔너리

    Returns:
        LangGraph 에이전트: MCP 도구가 연결된 React Agent
    """

    # MCP 클라이언트 생성 및 모든 서버의 도구를 통합 로딩
    client, tools = await setup_mcp_client(server_configs=server_configs)

    # OpenAI API (노트북 앞부분과 동일하게 OPENAI_API_KEY 사용)
    llm = ChatOpenAI(
        temperature=0,
        model="gpt-4.1",
        api_key=os.getenv("OPENAI_API_KEY"),
    )

    # React 패턴을 구현하는 에이전트 생성
    # - 추론(Reason): LLM이 상황을 분석하고 다음 행동 계획
    # - 행동(Act): MCP 도구를 호출하여 구체적 작업 실행
    agent = create_react_agent(
        llm, tools, checkpointer=InMemorySaver()  # 상태 저장을 위한 체크포인터
    )

    return agent

In [16]:
import sys
from pathlib import Path

# 날씨(stdio) 서버 스크립트 경로 — 앞쪽 로컬 MCP 셀과 동일하게 두 가지 cwd를 허용합니다.
_local = Path("02-MCP/server/mcp_server_local.py").resolve()
if not _local.is_file():
    _local = Path("server/mcp_server_local.py").resolve()
if not _local.is_file():
    raise FileNotFoundError(
        "mcp_server_local.py를 찾을 수 없습니다. mysuni_aiagent 또는 02-MCP 폴더를 연 뒤 실행하세요."
    )

# 다중 MCP 서버 구성 — 날씨(stdio) + 원격(streamable_http, 8002에서 기동된 서버)
server_configs = {
    "weather": {
        "command": sys.executable,
        "args": [str(_local)],
        "transport": "stdio",
    },
    "current_time": {
        "url": "http://127.0.0.1:8002/mcp",
        "transport": "streamable_http",
    },
}

# 다중 서버를 통합한 MCP React Agent 생성 (원격 서버 프로세스가 떠 있어야 합니다)
agent = await create_mcp_react_agent(server_configs)

✅ 3 개의 MCP 도구가 로드되었습니다:
  - get_weather
  - get_current_time
  - retrieve


/var/folders/fn/cyp1s2wj7c39s35jrjsksnkw0000gn/T/ipykernel_10896/3217752972.py:26: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [17]:
# React Agent 통합 테스트 - 다중 서버 활용
from langchain_core.runnables import RunnableConfig
from langchain_teddynote.messages import random_uuid, astream_graph

# 멀티 도구 테스트를 위한 대화 세션 생성
config = RunnableConfig(configurable={"thread_id": random_uuid()})

# 1차 테스트: 시간 정보 요청 (HTTP 서버 사용)
await astream_graph(
    agent, inputs={"messages": [("human", "현재 시간을 알려주세요")]}, config=config
)

# 2차 테스트: 날씨 정보 요청 (로컬 서버 사용)
# 동일한 세션을 사용하여 대화 맥락 유지
await astream_graph(
    agent,
    inputs={"messages": [("human", "현재 서울의 날씨도 알려주세요")]},
    config=config,  # 이전 대화와 연결된 동일한 세션 사용
)


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
2026-04-22 12:05:14 KST (UTC+0900)
🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
현재 시간은 2026년 4월 22일 12시 5분 14초(KST)입니다.
🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
It's always Sunny in 서울
🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
현재 서울의 날씨는 항상 맑음입니다.

{'node': 'agent',
 'content': AIMessageChunk(content='', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_26cb5ea9ef', 'service_tier': 'default'}, id='run--019db326-2b49-7bc0-b8c4-f1e29b1292a1'),
 'metadata': {'thread_id': '53daede5-37ab-4511-90de-a876976d41dc',
  'langgraph_step': 8,
  'langgraph_node': 'agent',
  'langgraph_triggers': ('branch:to:agent',),
  'langgraph_path': ('__pregel_pull', 'agent'),
  'langgraph_checkpoint_ns': 'agent:d6a0e182-2b15-0698-f7ca-a50a4ad9f3fa',
  'checkpoint_ns': 'agent:d6a0e182-2b15-0698-f7ca-a50a4ad9f3fa',
  'ls_provider': 'openai',
  'ls_model_name': 'gpt-4.1',
  'ls_model_type': 'chat',
  'ls_temperature': 0.0,
  'LANGSMITH_PROJECT': 'LangChain-Tutorial',
  'LANGSMITH_TRACING': 'true',
  'LANGSMITH_ENDPOINT': 'https://api.smith.langchain.com',
  'revision_id': 'aaaec88-dirty'}}

## 외부 MCP 서버에서 3rd Party 도구 사용하기

**Smithery AI란?**

- 사이트: https://smithery.ai/

- Smithery AI는 AI 에이전트 서비스의 허브 역할을 하는 플랫폼입니다. 

에이전트형 AI(예: 대형 언어 모델)가 외부 도구나 정보와 효율적으로 연결될 수 있도록 설계된 MCP 서버들을 검색하고 배포하는 역할을 수행합니다. 즉, AI가 다양한 외부 서비스와 손쉽게 통신할 수 있게 하는 중개자이자 생태계 허브입니다.

이 모든 서비스의 연결은 **MCP(Model Context Protocol)** 표준 프로토콜을 따릅니다. 이를 통해 AI가 검색, 프로그래밍, 파일 관리, 다양한 API 연동 등 여러 외부 기능을 하나의 허브(플랫폼)만으로 편리하게 활용할 수 있습니다

### 사전 설치 (npx)

1. 터미널을 엽니다.
2. 프로젝트 루트(`mysuni_aiagent`) 또는 `02-MCP` 폴더에서 작업합니다.
3. `npx --version`으로 설치 여부를 확인합니다.
4. Smithery API 키를 환경 변수로 설정합니다. (`SMITHERY_API_KEY`)

`npx`와 `SMITHERY_API_KEY` 준비 후 다음 셀을 실행하세요.

In [20]:
import os
import sys
from pathlib import Path
from shutil import which

# Smithery는 필수 사용
if which("npx") is None:
    raise EnvironmentError("npx를 찾을 수 없습니다. Node.js/npm 설치 후 다시 실행하세요.")

smithery_api_key = os.getenv("SMITHERY_API_KEY")
if not smithery_api_key:
    raise EnvironmentError("SMITHERY_API_KEY 환경 변수가 필요합니다.")

# 로컬 날씨 서버 경로 (프로젝트 루트 / 02-MCP 폴더 모두 지원)
local_script = Path("02-MCP/server/mcp_server_local.py").resolve()
if not local_script.is_file():
    local_script = Path("server/mcp_server_local.py").resolve()
if not local_script.is_file():
    raise FileNotFoundError("mcp_server_local.py를 찾을 수 없습니다.")

# Smithery 연결 설정 (최신 CLI: --key 옵션 미지원, env 전달 사용)
smithery_config = {
    "desktop-commander": {
        "command": "npx",
        "args": [
            "-y",
            "@smithery/cli@latest",
            "run",
            "@wonderwhy-er/desktop-commander",
        ],
        "env": {"SMITHERY_API_KEY": smithery_api_key},
        "transport": "stdio",
    }
}

# 1) Smithery 단독 연결 점검: 여기서 실패하면 키/권한/네트워크 문제
_, smithery_tools = await setup_mcp_client(server_configs=smithery_config)
print(f"Smithery 도구 로드 완료: {len(smithery_tools)}개")

# 2) 내부 + 외부 통합 구성
server_configs = {
    "weather": {
        "command": sys.executable,
        "args": [str(local_script)],
        "transport": "stdio",
    },
    "current_time": {
        "url": "http://127.0.0.1:8002/mcp",  # 원격 시간 서버(미리 기동 필요)
        "transport": "streamable_http",
    },
    **smithery_config,
}

# 3) 통합 MCP React Agent 생성 (Smithery 포함 필수)
mcp_app = await create_mcp_react_agent(server_configs)
print(f"MCP 에이전트 생성 완료 (servers: {list(server_configs.keys())})")

✅ 26 개의 MCP 도구가 로드되었습니다:
  - get_config
  - set_config_value
  - read_file
  - read_multiple_files
  - write_file
  - write_pdf
  - create_directory
  - list_directory
  - move_file
  - start_search
  - get_more_search_results
  - stop_search
  - list_searches
  - get_file_info
  - edit_block
  - start_process
  - read_process_output
  - interact_with_process
  - force_terminate
  - list_sessions
  - list_processes
  - kill_process
  - get_usage_stats
  - get_recent_tool_calls
  - give_feedback_to_desktop_commander
  - get_prompts
Smithery 도구 로드 완료: 26개
✅ 29 개의 MCP 도구가 로드되었습니다:
  - get_weather
  - get_current_time
  - retrieve
  - get_config
  - set_config_value
  - read_file
  - read_multiple_files
  - write_file
  - write_pdf
  - create_directory
  - list_directory
  - move_file
  - start_search
  - get_more_search_results
  - stop_search
  - list_searches
  - get_file_info
  - edit_block
  - start_process
  - read_process_output
  - interact_with_process
  - force_terminate
  - list

/var/folders/fn/cyp1s2wj7c39s35jrjsksnkw0000gn/T/ipykernel_10896/3217752972.py:26: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [21]:
# 새로운 대화 세션 생성
config = RunnableConfig(configurable={"thread_id": random_uuid()}, recursion_limit=100)

await astream_graph(
    mcp_app,
    inputs={
        "messages": [
            ("human", "현재 directory 를 조회하고, 폴더 리스트를 나열하세요. ")
        ]
    },
    config=config,  # 연속 대화 세션 유지
)


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
[DIR] .cache
[FILE] 01-MCP-Tools.ipynb
[DIR] assets
[FILE] install_node_npx.sh
[DIR] server
🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
현재 디렉토리에는 다음과 같은 폴더들이 있습니다:

- .cache
- assets
- server

필요한 폴더의 내용을 더 보고 싶으시면 말씀해 주세요!

{'node': 'agent',
 'content': AIMessageChunk(content='', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_036c525fce', 'service_tier': 'default'}, id='run--019db32a-e9c2-7d32-aad1-fc8b0c731e2f'),
 'metadata': {'thread_id': 'e87d4dc6-c9fc-49e0-8a37-cd6dae96a6fc',
  'langgraph_step': 3,
  'langgraph_node': 'agent',
  'langgraph_triggers': ('branch:to:agent',),
  'langgraph_path': ('__pregel_pull', 'agent'),
  'langgraph_checkpoint_ns': 'agent:4caac248-9e51-6f85-cacc-c567c97d4d90',
  'checkpoint_ns': 'agent:4caac248-9e51-6f85-cacc-c567c97d4d90',
  'ls_provider': 'openai',
  'ls_model_name': 'gpt-4.1',
  'ls_model_type': 'chat',
  'ls_temperature': 0.0,
  'LANGSMITH_PROJECT': 'LangChain-Tutorial',
  'LANGSMITH_TRACING': 'true',
  'LANGSMITH_ENDPOINT': 'https://api.smith.langchain.com',
  'revision_id': 'aaaec88-dirty'}}

In [23]:
await astream_graph(
    mcp_app,
    inputs={
        "messages": [
            (
                "human",
                "01-MCP-Tools.ipynb 파일을 읽고 내용을 정리하여 02-MCP 폴더 하위에 summary.txt 파일로 저장해 주세요.",
            )
        ]
    },
    config=config,
)


🔄 Node: agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
[Reading 1000 lines from start (total: 1217 lines, 217 remaining)]

{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# MCP(Model Context Protocol)\n",
        "\n",
        "### MCP(Model Context Protocol) 정의\n",
        "\n",
        "**MCP**는 **AI 애플리케이션과 외부 도구 간의 상호작용을 표준화**한 오픈 프로토콜입니다.  \n",
        "즉, LLM이 다양한 외부 시스템(API, 데이터베이스, 도구 등)과 일관된 방식으로 통신할 수 있도록 하는 **표준 인터페이스 계층**입니다.\n",
        "\n",
        "---\n",
        "\n",
        "### MCP의 4가지 핵심 특징\n",
        "\n",
        "| 특징 | 설명 | 장점 | 예시 |\n",
        "|------|------|------|------|\n",
        "| **표준화된 도구 인터페이스** | 모든 외부 도구가 동일한 프로토콜로 연결 | 새로운 도구 추가 시 별도 학습 불필요 | 날씨 API, DB, 파일 시스템 등 동일 방식 호출 |\n",
        "| **다양한 전송 메커니즘** | **공식: `stdio`, `Streamable HTTP` (구 `HTTP+SSE`는 deprecated)** | 환경(로컬/원격)에 맞는 최적 방식 선택 | 로컬은 `stdio`, 원격은 

{'node': 'agent',
 'content': AIMessageChunk(content='', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_26cb5ea9ef', 'service_tier': 'default'}, id='run--019db32c-a47a-7140-a527-e8214e59baa4'),
 'metadata': {'thread_id': 'e87d4dc6-c9fc-49e0-8a37-cd6dae96a6fc',
  'langgraph_step': 17,
  'langgraph_node': 'agent',
  'langgraph_triggers': ('branch:to:agent',),
  'langgraph_path': ('__pregel_pull', 'agent'),
  'langgraph_checkpoint_ns': 'agent:a4bd16de-d889-4299-1fcd-c63a278fa82b',
  'checkpoint_ns': 'agent:a4bd16de-d889-4299-1fcd-c63a278fa82b',
  'ls_provider': 'openai',
  'ls_model_name': 'gpt-4.1',
  'ls_model_type': 'chat',
  'ls_temperature': 0.0,
  'LANGSMITH_PROJECT': 'LangChain-Tutorial',
  'LANGSMITH_TRACING': 'true',
  'LANGSMITH_ENDPOINT': 'https://api.smith.langchain.com',
  'revision_id': 'aaaec88-dirty'}}